# Retrieval-Augmented Generation with LLMs: Adding External Databases via RAG

This notebook is an English, GitHub-friendly translation of the original document.  
The Python code is kept unchanged.

## What is RAG?

Retrieval-Augmented Generation (RAG) is a basic technique that adds external data and databases to an LLM.  
With RAG, an LLM can skip continuous retraining and obtain the latest information.

Earlier sections in the original article discussed:

1. LLM training formats, fine-tuning (LoRA), and multi-GPU training (FSDP)
2. Extending context length with RoPE (and strengthening it with NTK), while making input length variable
3. Converting an LLM (a text model) into a Vision-Language Model (ViT-style) through CLIP

Each of those topics came with runnable code.

MoE is a modification to the MLP layer and is an application of standard theorems from higher mathematics; its principle is beyond the scope of compulsory education.  
With RAG technology added to the mix, even high school students can, by simply copying and pasting, download open-source model parameters and obtain model metrics close to those of leading companies.

This also benefits from the support of open-source communities by companies such as DeepSeek, OpenAI, Microsoft, Meta, Claude, and Alphabet, as well as the infrastructure provided by compute platforms such as ......

## How RAG works

RAG uses a model, which can be an Encoder model, a Decoder model, or other Transformer-based models.  
For RAG, “Encoder” and “Decoder” are not strictly accurate translations here. In the RAG context, an encoder usually refers to a model trained with bidirectional masking, while a decoder refers to a model trained with causal masking. These names mainly come from the historical development of Transformers.

Typically, a set of input text tokens is converted by a tokenizer into a T×D-shaped representation. After several iterations, it remains a T×D data structure.  
Then a pooling layer reduces it to a one-dimensional vector, usually still of size D.  
Finally, a projection operator maps that one-dimensional vector to another one-dimensional vector of size K.

RAG is the framework/architecture. The component used to accomplish this is called an embedding model.  
(“Embedding” is also a term with multiple meanings.)

In practical use, each file in the database is turned into one or more vectors. These vectors are matched against the vectors generated from the input.  
The most relevant vectors, for example those with the highest dot product, are retrieved, and the corresponding text is added to the LLM’s input so the model can process it.

For example, an input may be: “What did I eat yesterday?”  
However, the LLM’s training data may not include “yesterday” in a personal context, but that information may exist in an Alipay bill.  
Given this input, the embedding model generates a vector and matches it against every entry, including Alipay records.  
Then the text mapped from the highest-scoring vector(s) is added to the LLM input:

`{user: "What did I eat yesterday?"} {assistant: "context: restaurant receipt xxx, braised fish ×1, ... "}`

Next, the LLM produces a real-time answer based on the user input plus the context added through RAG.

It is important to note that an embedding model only converts large amounts of text into one-dimensional vectors of equal length.  
This often produces a very large number of vectors with the same dimension, which makes search and matching difficult.  
For a long text file, such as a book with more than a thousand pages, the system may generate tens of thousands of excerpts from each paragraph, which creates significant retrieval pressure.

In industry and research, specialized vector search algorithms are usually used.  
In research, the FAISS library (Facebook AI Similarity Search) is often used, although it may perform less well in industry.

In particular, the claim often seen in Chinese textbooks that one book should be grouped into chunks of 100 pages is not correct in industry.  
Some leading Western models measured a few years ago could precisely index an entire book or hundreds of pages, far beyond their context length, and provide exact guidance down to each page, sentence, or even formula. That corresponds to more than 5 million context length and 10,000 complex mathematical equations. This was far beyond the technical specifications of LLMs at the time and beyond the capacity of the agent systems of that era; what was used was likely a RAG-like approach.

My personal guess is that this may be one reason why, compared with similarly capable domestic models, Western large models are still used in industry with a RAM-like size even when they have far less actual storage.  
By contrast, due to technical reasons, domestic large models may have heavier model architectures that occupy too much RAM, making it hard to load more fine-grained vectors. That may put them at a disadvantage in industrial RAG applications, although open-source domestic RAG embedding models are now among the leading tier, possibly for similar reasons.

## After the theory: a direct example of an embedding model

The following code demonstrates a simple embedding model workflow.


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# ------------------------------------------------------------
# 1. Load Model (assuming local path or Hugging Face ID)
# ------------------------------------------------------------
MODEL_PATH = "/root/private_data/ARG_microsoft/multilingual-e5-large"  # or 'intfloat/multilingual-e5-large'

from huggingface_hub import snapshot_download

snapshot_download(repo_id="intfloat/multilingual-e5-large", 
                  # local_dir="/root/private_data/ARG_microsoft/multilingual-e5-large"
                  local_dir=MODEL_PATH
                 )


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModel.from_pretrained(MODEL_PATH)
model.eval()

# Optional: Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# ------------------------------------------------------------
# 2. Embedding Function (as defined)
# ------------------------------------------------------------
def get_embedding(text, prefix="query: "):
    input_text = prefix + text
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        attention_mask = inputs["attention_mask"]
        token_embeddings = outputs.last_hidden_state

        # Mean pooling with mask
        mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        pooled = sum_embeddings / sum_mask

        # Normalize to unit length
        normalized = F.normalize(pooled, p=2, dim=1)

    return normalized

# ------------------------------------------------------------
# 3. Define a Corpus of Documents
# ------------------------------------------------------------
documents = [
    "Returns must be initiated within 30 days of purchase. Items must be unworn with original tags attached.",
    "We offer free standard shipping on all orders over $50. Expedited shipping is available for an additional fee.",
    "Our privacy policy outlines how we collect, use, and protect your personal information.",
    "To reset your password, click the 'Forgot Password' link on the login page and follow the instructions sent to your email.",
    "We accept Visa, Mastercard, American Express, and PayPal. All transactions are secured with SSL encryption.",
]

# ------------------------------------------------------------
# 4. Encode All Documents (with "passage: " prefix)
# ------------------------------------------------------------
print("Encoding documents...")
doc_embeddings = []
for doc in documents:
    emb = get_embedding(doc, prefix="passage: ")
    doc_embeddings.append(emb)

# Stack into a single tensor: (num_docs, 1024)
doc_embeddings = torch.cat(doc_embeddings, dim=0)  # (5, 1024)

# ------------------------------------------------------------
# 5. User Query
# ------------------------------------------------------------
query = "How do I return an item I bought?"
print(f"\nQuery: '{query}'")

# Encode query (with "query: " prefix)
query_emb = get_embedding(query, prefix="query: ")  # (1, 1024)

# ------------------------------------------------------------
# 6. Compute Similarities (Dot Product)
# ------------------------------------------------------------
# Since vectors are normalized, dot product = cosine similarity
similarities = torch.matmul(query_emb, doc_embeddings.T).squeeze(0)  # (5,)

# Convert to Python list for easy viewing
scores = similarities.cpu().tolist()

# ------------------------------------------------------------
# 7. Display Results
# ------------------------------------------------------------
print("\n--- Similarity Scores ---")
for i, (doc, score) in enumerate(zip(documents, scores)):
    print(f"Doc {i+1}: {score:.4f} | {doc[:60]}...")

# Find the best match
best_idx = torch.argmax(similarities).item()
print(f"\n--- Top Match (Score: {scores[best_idx]:.4f}) ---")
print(documents[best_idx])


## Notes

The original closing contact information and other closing text have been omitted from the notebook because they are not part of the technical explanation.  
If needed, they can be added back as a final markdown cell.
